In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd

import src.attn_tracking_lightning as lm 
importlib.reload(lm)
import yaml
from pytorch_lightning import Trainer




os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"




In [2]:
!nvidia-smi

Fri Jul 21 19:38:54 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.106.00   Driver Version: 460.106.00   CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  RTX A6000           On   | 00000000:AF:00.0 Off |                  Off |
| 30%   36C    P8    21W / 300W |     13MiB / 48685MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [3]:
config_path = "config/attentional_cue/attn_modern_cnn_speech_and_noise.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['data']['loader']['num_workers'] = 5
config['data']['loader']['batch_size'] = 64
config['data']['audio']['rep_kwargs']['rep_on_gpu'] = True



In [4]:
import pickle

class_dict = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" ))

In [5]:
# len(class_dict)
ix_to_word = {v:k for k,v in class_dict.items()}

In [6]:
module = lm.AttentionalTrackingModule(config)

ln_first
Using ModernCNN
num_classes={'num_words': 794}
Model performing word task
self.output_height=5
self.output_len=7
self.n_layers=18
center_crop=False
binaural=False
using FIR cochleagram


In [7]:
# dataset = module.dataset(**config['corpus'], modee='train')

In [8]:
trainer = Trainer(
    precision=32,
    # default_root_dir='test_log_dump/',
    # val_check_interval=1000,
#     limit_train_batches=0.,
    limit_val_batches=0.0,
    num_nodes=1,
    gpus=1,
    # detect_anomaly=True,
    accelerator="gpu",
)
trainer.fit(module)

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/loops/utilities.py:91: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                  | Type                   | Params
-----------------------------------------------------------------
0 | bg_combine_transforms | AudioCompose           | 0     
1 | audio_transforms      | AudioCompose           | 0     
2 | model                 | AttnSequentialAttacker | 14.9 M
3 | loss_fn               | CrossEntropyLoss       | 0     
4 | train_acc             | Accuracy               | 0     
5 | valid_acc             | Accuracy               | 0     
6 | test_acc              | Accuracy               | 0     

<class 'corpus.jsinV3_attn_multi_talker_w_audioset.jsinV3_attn_multi_talker_w_audioset'>


Training: 0it [00:00, ?it/s]

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/trainer.py:726: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")


In [ ]:
!nvidia-smi

Fri Jul 21 19:07:29 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 515.86.01    Driver Version: 515.86.01    CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:02:00.0 Off |                  N/A |
| 34%   54C    P2    74W / 250W |   7643MiB / 11264MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------